In [1]:
!git clone https://github.com/dcbisson23/CISC839_A2
!git clone https://github.com/taoyds/test-suite-sql-eval

Cloning into 'CISC839_A2'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 95 (delta 45), reused 94 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 297.98 KiB | 9.31 MiB/s, done.
Resolving deltas: 100% (45/45), done.
Cloning into 'test-suite-sql-eval'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 61 (delta 20), reused 16 (delta 16), pack-reused 31 (from 2)
Receiving objects: 100% (61/61), 619.62 KiB | 8.61 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [2]:
from transformers import set_seed
set_seed(20230334)

In [3]:
# This is mostly pulled from A1's py file, for ease of use.

TABLES_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/tables.json"
QUESTIONS_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev.json"
GOLD_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev_gold.sql"
DATABASE_PATH = "/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"

# This is where model data (generation , formatted prompts, responses) are written to/read from.
DATA_ROOT = "/kaggle/working/CISC839_A2/data"
GEN_CONFIG_FILENAME = "generation_config.json"
PROMPTS_FILENAME = "formatted_prompts.jsonl"
RESPONSES_FILENAME = "responses.jsonl"
PREDICTIONS_FILENAME = "predictions.txt"
GOLD_FILENAME = "gold.txt"

SAMPLE_RATE = 0.125

# This is the system context used for prompting.
# NOTE: Modified from A1, removed clause restricting use of aliasing in query (found many gold examples used aliasing where A1 models did not).
SYS_CONTEXT = ("""You are an expert SQLite developer, tasked with writing a valid SQL query to answer a user question using a given SQL schema.
The user will provide the following information:
    1. The database schema as a standard SQLite DDL script, and
    3. The question to be answered by querying this database.
Your response must adhere to the following guidelines:
    1. (CRITICAL) You must provide ONLY a valid SQLite query to answer the provided question, and nothing else.
    2. Use ONLY the tables and columns provided in the schema.
    3. Do not use aliasing (x AS y) unless strictly necessary.
""")

# This is the model being used for A2.
MODEL = {
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct"
         }

In [4]:
import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(f"{DATA_ROOT}/{MODEL['model']}", exist_ok=True)

In [5]:
from CISC839_A2.scripts.utils import load_tokenizer, unload_model
from CISC839_A2.scripts.generate_prompts import SpiderPromptBuilder

builder = SpiderPromptBuilder(TABLES_PATH, QUESTIONS_PATH, sys_context=SYS_CONTEXT)

tokenizer = load_tokenizer(MODEL)

prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"

builder.generate_prompts(prompts_path, tokenizer, sample_rate=SAMPLE_RATE)

unload_model(tokenizer)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Generating prompts: 1034it [00:00, 40513.32it/s]


Successfully generated 135 prompts to /kaggle/working/CISC839_A2/data/Qwen/Qwen2.5-Coder-3B-Instruct/formatted_prompts.jsonl


In [6]:
from CISC839_A2.scripts.utils import load_model, unload_model
from CISC839_A2.scripts.get_predictions import LLMPrompter

print("Loading model "+MODEL['model'])
model = load_model(MODEL)
tokenizer = load_tokenizer(MODEL)

gen_path = f"{DATA_ROOT}/{MODEL['model']}/{GEN_CONFIG_FILENAME}"
prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"
responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"

prompter = LLMPrompter(model, tokenizer)
prompter.load_gen_config(gen_path)
prompter.run_prompt_pipeline(prompts_path, responses_path)

unload_model(model)
unload_model(tokenizer)

Loading model Qwen/Qwen2.5-Coder-3B-Instruct


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generating responses: 100%|██████████| 135/135 [06:59<00:00,  3.10s/it]


Successfully wrote 135 responses to /kaggle/working/CISC839_A2/data/Qwen/Qwen2.5-Coder-3B-Instruct/responses.jsonl


In [7]:
import json
from CISC839_A2.scripts.utils import load_tokenizer, get_data_from_response
responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"
predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"

with open(responses_path) as f:
    responses = f.readlines()
with open(GOLD_PATH) as f:
    gold_sql = f.readlines()


predictions = []
golds = []
reasoning_traces = []

for response_json in responses:
    response_dict = json.loads(response_json)
    id = response_dict["id"]
    db_id = response_dict["db_id"]
    response = response_dict["response"]
    response_data = get_data_from_response(response)
    pred = response_data["prediction"]
    gold = gold_sql[id]
    predictions.append(pred)
    # To match needed formatting for evaluation script.
    golds.append(gold)

with open(gold_path, "w", encoding="utf-8") as f_out:
    f_out.write("".join(golds))
with open(predictions_path, "w", encoding="utf-8") as f_out:
    f_out.write("\n".join(predictions))

In [8]:
import os
import sys
cwd = os.getcwd()
sys.path.insert(0, f"{cwd}/test-suite-sql-eval")

In [1]:
# Run this for a deeper score analysis on a question-by-question basis.
from process_sql import Schema, get_schema, get_sql
from evaluation import Evaluator
from exec_eval import eval_exec_match
import json
import csv

import nest_asyncio
nest_asyncio.apply()

predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
database_path = f"/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"

with open(gold_path, "r") as f:
    # Took this code from evaluation.py.evaluate.
    glist = []
    gseq_one = []
    for l in f.readlines():
        if len(l.strip()) == 0:
            glist.append(gseq_one)
            gseq_one = []
        else:
            lstrip = l.strip().split('\t')
            gseq_one.append(lstrip)
    if len(gseq_one) != 0:
        glist.append(gseq_one)
    # End of block
# Not sure why but they nest glist in another list so I'm changing that here.
glist = glist[0]
evaluator = Evaluator()
results = []
with open(predictions_path, "r") as f:
    for idx, p_str in enumerate(f.readlines()):
        # Can't use question id for this since using gold
        g = glist[idx]
        # More taken from eval script.
        g_str, db_id = g
        db = os.path.join(database_path, db_id, db_id + ".sqlite")
        schema = Schema(get_schema(db))
        g_sql = get_sql(schema, g_str)
        hardness = evaluator.eval_hardness(g_sql)
        score = eval_exec_match(
            db=db,
            p_str=p_str,
            g_str=g_str,
            plug_value=False,
            keep_distinct=False,
            progress_bar_for_each_datapoint=False
        )
        result = {
            "id": idx,
            "db_id": db_id,
            "hardness": hardness,
            "score": score,
        }
        results.append(result)

results_path = f"{DATA_ROOT}/{MODEL['model']}/full_results.csv"
with open(results_path, "w", encoding="utf-8", newline="") as f_out:
    fieldnames = ["id", "db_id", "hardness", "score"]
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)


ModuleNotFoundError: No module named 'process_sql'